[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Sending Data


## What you will be able to do

Send a JSON body with `POST`, `PUT` and `PATCH`, remove a resource with `DELETE`, read the `201`,
`204`, `415` and `422` that come back, say which of those requests are safe to send twice, and make a
`POST` safe to send again with an idempotency key.


## The idea

### The problem

Every request so far read something. An API is also how a program changes things: an app places an
order, a script adds a record, a form saves an edit. A request that changes something carries data,
in a body, and uses a method that says what the server should do with it.

Changing things raises a question that reading never did: what happens when the same request
arrives twice? A retry after a timeout sends it twice, and so do a double click and a script run
again. Reading a station twice changes nothing. Adding a station twice leaves two, and nothing about
the second request looks wrong. The practice API's stations are read-only, so this notebook changes
something else: `/network/plans`, the stations the network plans to build.

### What sending data is

> A request that changes something sends its data in a **body**, and names the kind of data in
> `Content-Type`, most often `application/json`. **`POST`** asks the server to create something new
> and to choose its address, which the `201 Created` response names in `Location`. **`PUT`** replaces
> what is at an address with the body sent, **`PATCH`** changes only the parts the body names, and
> **`DELETE`** removes what is at an address. A method is **idempotent** when sending the same request
> twice leaves the server as sending it once does: `PUT` and `DELETE` are, `POST` is not, and `PATCH`
> is not promised to be.

### Why it works that way

- **The body carries the data, and `Content-Type` says what it is.** A server reads a body the way its
  `Content-Type` says, and refuses one it cannot read with `415 Unsupported Media Type`. requests'
  `json=` writes a body as JSON, and sets the header.
- **`POST` goes to a collection, and the server names what it creates.** A client cannot know the
  address of something that does not exist yet, so the response tells it, in `Location`.
- **`PUT` and `PATCH` go to the thing itself.** A `PUT` body is the whole thing, so a field left out is
  gone. A `PATCH` body is only the change.
- **Idempotent means safe to send twice, not the same answer twice.** A second `DELETE` gets `404`
  where the first got `204`, and the plan is gone either way. A second `POST` gets a second `201`, and
  makes a second plan.
- **A lost response is the hard case.** A timeout on a `POST` can come after the server created the
  plan, so the client cannot tell whether sending it again would leave one plan or two. An
  **idempotency key**, a random value sent in a header and the same on every attempt, lets the server
  recognize the repeat and answer it with its first result.
- **A change can be made from a stale copy.** Two clients that read a plan and write it back can undo
  each other's changes. `If-Match`, carrying the `ETag` from the read, makes the server refuse a write
  to a plan that has changed since, with `412 Precondition Failed`.

### Where you will meet this

GitHub's API stars a repository with `PUT /user/starred/{owner}/{repo}` and removes the star with
`DELETE`, and both answer `204 No Content`: starring a repository twice leaves it starred once.
Stripe's API accepts an `Idempotency-Key` header on every `POST`, saves the status code and body of
the first request made with a key, and returns them for a repeat, so that a request sent again after
a connection error creates one object, not two. It suggests a version 4 UUID for each key, and
answers a repeat whose parameters differ from the first request's with an error. Later in this guide,
the **Validating Requests** notebook sends a `422` from the server's side, as the practice API does
here.

### What this notebook covers

- `POST` with a JSON body, and `201 Created` with a `Location`
- What `json=` puts in a request, and the same request sent with curl
- A body the server refuses, with `422` and a list of problems
- `PATCH` to change part of a plan, `PUT` to replace all of it, and `DELETE`, with `204 No Content`
- Which methods are idempotent, shown by sending each request twice
- A `POST` whose response was lost, and the duplicate that sending it again made
- An `Idempotency-Key` that makes a `POST` safe to send again
- A client that creates, changes and removes plans, and makes every retry safe
- Five errors, from a body sent as a form to a change undone by a stale copy

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import requests

plan = {"name": "Narvik", "latitude": 68.44, "longitude": 17.43}
response = requests.post("http://127.0.0.1:8765/network/plans", json=plan, timeout=10)

print(response.status_code, response.reason, "| Location:", response.headers["Location"])
print(response.json())
```

```
201 Created | Location: /network/plans/1
{'id': 1, 'name': 'Narvik', 'latitude': 68.44, 'longitude': 17.43}
```

One request with a body, one new plan on the server, and its address in the response.


## Setup

Ten imports, the last of them the practice API.

- `requests` sends every request, with `json=` for a body, and `requests.request` for any method
- `uuid` makes an idempotency key, with `uuid4`
- `random` chooses a wait between a client's attempts, from a seeded generator
- `time` waits between those attempts
- `date` is a date, which JSON has no way to hold
- `urllib.request` fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here
- `sys` tells this cell whether the notebook is running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, started in the background by `start()`

`/network/plans` is empty whenever this cell runs. Every cell below that sends a `POST` creates a
plan, so running a cell a second time gives higher ids than the ones printed here, which is the
behavior this notebook is about.


In [1]:
import importlib
import random
import sys
import time
import urllib.request
import uuid
from datetime import date
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
print("The practice API is running at", BASE)


The practice API is running at http://127.0.0.1:8765


## Worked examples

### Sending a body: POST

`/network/plans` holds the stations the network plans to build, and unlike the stations themselves,
a client can add, change and remove plans. `json=` sends a dictionary as the request's body. This
plan's latitude is mistyped, and a later section corrects it:


In [2]:
narvik = {"name": "Narvik", "latitude": 86.44, "longitude": 17.43}
response = requests.post(f"{BASE}/network/plans", json=narvik, timeout=10)

print(response.status_code, response.reason)
print("Location:", response.headers["Location"])
print(response.json())
narvik_url = f"{BASE}{response.headers['Location']}"


201 Created
Location: /network/plans/1
{'id': 1, 'name': 'Narvik', 'latitude': 86.44, 'longitude': 17.43}


`201 Created` says the request made something new, and `Location` gives its address, which the
server chose, since the client could not know it before the plan existed. The body is the plan as the
server stored it, with the `id` it added, and `narvik_url` keeps the address for later. Code that
checks `status_code == 200` would count this success as a failure, which is why the **Status Codes**
notebook checked the class of a code instead.

### What json= sends, and the same request from curl

The request that requests sent stays attached to the response:


In [3]:
sent = response.request
print(sent.method, sent.url)
print("Content-Type:", sent.headers["Content-Type"])
print(sent.body)


POST http://127.0.0.1:8765/network/plans
Content-Type: application/json
b'{"name": "Narvik", "latitude": 86.44, "longitude": 17.43}'


`json=` wrote the dictionary as JSON text, sent it as bytes, and set `Content-Type: application/json`,
the header the **Headers and Content Types** notebook said a client sets on a body it sends. curl
sends the same request with `-H` for the header and `-d` for the body, the two flags the **Exploring
an API** notebook named. `-d` makes curl send `POST`, and because IPython reads `{BASE}` in a `!` line
as a Python name, the JSON's own braces are doubled:


In [4]:
!curl -s -i {BASE}/network/plans -H "Content-Type: application/json" -d '{{"name": "Alta", "latitude": 69.97, "longitude": 23.27}}'


HTTP/1.1 201 Created
Server: PracticeAPI/1.0
Date: Sun, 01 Mar 2026 09:00:00 GMT
Content-Type: application/json
Content-Length: 64
Location: /network/plans/2
ETag: "7eefdf5809152d9f"

{"id": 2, "name": "Alta", "latitude": 69.97, "longitude": 23.27}

`-i` printed the status line and the headers above the body, so the second plan's `Location` is
there, and so is an `ETag`, which the **Headers and Content Types** notebook used to skip a download,
and which Common errors uses to guard a change.

### A body the server refuses: 422

A plan with a field missing, or a value out of range, is refused before anything is saved:


In [5]:
lakselv = {"name": "Lakselv", "latitude": 170.05}          # no longitude, and a latitude beyond 90
response = requests.post(f"{BASE}/network/plans", json=lakselv, timeout=10)

print(response.status_code, response.reason, "|", response.json()["error"])
for problem in response.json()["problems"]:
    print(f"  {problem['field']}: {problem['problem']}")


422 Unprocessable Content | the plan has problems
  latitude: must be a number from -90 to 90
  longitude: is required


`422` means the server read the body and found it wrong, and this one says where, with a problem for
each field. Nothing was saved, so the client corrects the body and sends it again. `raise_for_status`
raises for a `422`, as for any `4xx`, but the problems are in the body, not in the exception's
message, so a client reads the body before it raises.

### Changing a plan: PATCH and PUT

`PATCH` changes only the fields its body names, which suits a correction, and `PUT` replaces the whole
plan with its body. Here `PATCH` corrects Narvik's latitude, and `PUT` then sends the whole plan with
an elevation added:


In [6]:
patched = requests.patch(narvik_url, json={"latitude": 68.44}, timeout=10)
print("PATCH", patched.status_code, patched.json())

whole = {"name": "Narvik", "latitude": 68.44, "longitude": 17.43, "elevation_m": 8}
replaced = requests.put(narvik_url, json=whole, timeout=10)
print("PUT  ", replaced.status_code, replaced.json())


PATCH 200 {'id': 1, 'name': 'Narvik', 'latitude': 68.44, 'longitude': 17.43}
PUT   200 {'id': 1, 'name': 'Narvik', 'latitude': 68.44, 'longitude': 17.43, 'elevation_m': 8}


The `PATCH` body named one field, and the other three stayed as they were. The `PUT` body was the
whole plan, and the plan became exactly that body. Both answered `200` with the plan as it now is. A
`PUT` that leaves a field out removes it, which Common errors shows, and a `PATCH` that sends a field
as `null` removes just that field. These are the two rows of the **What an API Is** notebook's table
that sounded alike: replace a resource, or change part of it.

### Removing a plan: DELETE

`DELETE` needs no body:


In [7]:
created = requests.post(f"{BASE}/network/plans", json={"name": "Kirkenes", "latitude": 69.73, "longitude": 30.05}, timeout=10)
kirkenes_url = f"{BASE}{created.headers['Location']}"

removed = requests.delete(kirkenes_url, timeout=10)
print("DELETE", removed.status_code, removed.reason, "| body:", removed.content)
print("GET   ", requests.get(kirkenes_url, timeout=10).status_code)
print("DELETE", requests.delete(kirkenes_url, timeout=10).status_code)


DELETE 204 No Content | body: b''
GET    404
DELETE 404


`204 No Content` confirms the removal and sends no body, as the `204` in the **Status Codes** notebook
did, so there is nothing for `json()` to read. After it, a `GET` finds nothing, and a second `DELETE`
finds nothing to remove.

### Idempotent, and not

Sending each request twice shows which methods are idempotent. `twice` sends one request two times,
and prints both status codes and how the number of plans changed. `requests.request` takes the method
as its first argument, so one function can send all four:


In [8]:
def plan_count():
    return len(requests.get(f"{BASE}/network/plans", timeout=10).json())


def twice(method, url, **kwargs):
    """Send one request two times, and print both status codes and the change in the number of plans."""
    before = plan_count()
    responses = [requests.request(method, url, timeout=10, **kwargs) for _ in range(2)]
    print(f"{method:<6} {responses[0].status_code}, {responses[1].status_code} | plans: {plan_count() - before:+d}")
    return responses


posted = twice("POST", f"{BASE}/network/plans", json={"name": "Andenes", "latitude": 69.32, "longitude": 16.12})
andenes_url = f"{BASE}{posted[0].headers['Location']}"
for method, body in [("PUT", {"name": "Andenes", "latitude": 69.32, "longitude": 16.12, "elevation_m": 10}),
                     ("PATCH", {"elevation_m": 12}), ("DELETE", None)]:
    twice(method, andenes_url, json=body)


POST   201, 201 | plans: +2
PUT    200, 200 | plans: +0
PATCH  200, 200 | plans: +0
DELETE 204, 404 | plans: -1


Two `POST`s made two plans for Andenes: the server had no way to tell that the second was meant as
the first again. Two `PUT`s left one plan holding the body, and so did two `PATCH`es, because this
`PATCH` sets a field to a value. A `PATCH` that adds to a value, such as one that raises a count,
changes the plan again with every repeat, which is why HTTP does not promise that `PATCH` is
idempotent. The second `DELETE` got `404`, a different answer, and left the server as the first did.
Idempotent describes what the server is left with, not what it answers.

### A response that never arrived

A repeated `POST` does most harm when the client cannot tell whether the first one worked. The
practice API accepts `delay`, a number of seconds to wait before it answers a `POST` it has already
carried out, so that a notebook can lose a response on purpose. With `delay=2` and a read timeout of
1 second, the client gives up before the answer comes, and sends the request again:


In [9]:
hammerfest = {"name": "Hammerfest", "latitude": 70.66, "longitude": 23.68}
before = plan_count()
try:
    requests.post(f"{BASE}/network/plans", params={"delay": 2}, json=hammerfest, timeout=1)
except requests.exceptions.ReadTimeout:
    print("no response, so the plan is sent again")
    requests.post(f"{BASE}/network/plans", json=hammerfest, timeout=10)

names = [plan["name"] for plan in requests.get(f"{BASE}/network/plans", timeout=10).json()]
print("plans added:", plan_count() - before, "| plans named Hammerfest:", names.count("Hammerfest"))


no response, so the plan is sent again
plans added: 2 | plans named Hammerfest: 2


The first request timed out after the server had created the plan, and sending it again created
another. The client treated the timeout as the **Errors and Retries** notebook's retry loop treats one,
and for a `POST` that is wrong: from the client's side, a lost response and a failed request look the
same.

### An Idempotency-Key: one plan however often it is sent

An idempotency key makes a `POST` safe to send again. The client makes one random key for each plan it
means to create, with `uuid.uuid4()`, and sends it in an `Idempotency-Key` header on every attempt at
that plan. The server saves the result of the first request with each key, and a repeat with the same
key and body gets that result again, marked `Idempotent-Replayed: true`, and creates nothing:


In [10]:
harstad = {"name": "Harstad", "latitude": 68.80, "longitude": 16.54}
key = {"Idempotency-Key": str(uuid.uuid4())}
before = plan_count()
try:
    requests.post(f"{BASE}/network/plans", params={"delay": 2}, json=harstad, headers=key, timeout=1)
except requests.exceptions.ReadTimeout:
    print("no response, so the plan is sent again, with the same key")
    again = requests.post(f"{BASE}/network/plans", json=harstad, headers=key, timeout=10)
    print(again.status_code, "| Idempotent-Replayed:", again.headers["Idempotent-Replayed"], "|", again.json()["name"])

print("plans added:", plan_count() - before)


no response, so the plan is sent again, with the same key
201 | Idempotent-Replayed: true | Harstad
plans added: 1


The second request got the `201` of the first, whose response had been lost, and the server created
nothing new. A key names one request, so sending it with a different body is refused, since the server
cannot tell which of the two was meant:


In [11]:
reused = requests.post(f"{BASE}/network/plans", json={**harstad, "elevation_m": 20}, headers=key, timeout=10)
print(reused.status_code, reused.json()["error"])


422 this Idempotency-Key was already used with a different body


Stripe's API works this way for every `POST`, and suggests a version 4 UUID for each key, which is what
`uuid.uuid4()` makes. A body that fails validation is not saved under its key, so a client can correct
the body and send it again with the same key. `PUT` and `DELETE` need no key, because repeating them is
already safe.

### A client that changes plans safely

The pieces of this notebook, in one client. `PlansClient` sends every body as JSON, and sends a request
again after a timeout, a lost connection or a `5xx`, as the **Errors and Retries** notebook did. It
makes each of those retries safe: a new plan carries one idempotency key across all its attempts, a
change sets values, so sending it twice does no harm, and a removal counts a `404` as done, since an
earlier attempt may have removed the plan already:


In [12]:
class PlansClient:
    """Creates, changes and removes plans, sending a request again when that is safe, and never creating a plan twice."""

    def __init__(self, base, attempts=3, timeout=(3.05, 5), seed=None):
        self.base = base
        self.attempts = attempts
        self.timeout = timeout
        self.rng = random.Random(seed)
        self.log = []

    def send(self, method, path, **kwargs):
        """The response to a request, sent again after a timeout, a lost connection or a 5xx."""
        for attempt in range(1, self.attempts + 1):
            try:
                outcome = requests.request(method, f"{self.base}{path}", timeout=self.timeout, **kwargs)
                self.log.append(f"{method} {path}, attempt {attempt}: {outcome.status_code}")
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as error:
                outcome = error
                self.log.append(f"{method} {path}, attempt {attempt}: {type(error).__name__}")
            failed = isinstance(outcome, Exception) or outcome.status_code >= 500
            if not failed or attempt == self.attempts:
                break
            time.sleep(self.rng.uniform(0, 2 ** (attempt - 1)))
        if isinstance(outcome, Exception):
            raise outcome
        return outcome

    def create(self, plan, **params):
        """A new plan, created once however many attempts it takes."""
        key = {"Idempotency-Key": str(uuid.uuid4())}           # one key for every attempt at this plan
        response = self.send("POST", "/network/plans", json=plan, params=params, headers=key)
        response.raise_for_status()
        return response.json()

    def change(self, plan_id, **fields):
        """The plan, with the fields given set to their new values."""
        response = self.send("PATCH", f"/network/plans/{plan_id}", json=fields)
        response.raise_for_status()
        return response.json()

    def remove(self, plan_id):
        """Remove a plan. A 404 means it is gone, perhaps removed by an earlier attempt."""
        response = self.send("DELETE", f"/network/plans/{plan_id}")
        if response.status_code not in (204, 404):
            response.raise_for_status()


client = PlansClient(BASE, timeout=(3.05, 1), seed=7)
before = plan_count()
plan = client.create({"name": "Karasjok", "latitude": 69.47, "longitude": 25.51}, delay=2)
print("created:", plan["name"], "| plans added:", plan_count() - before)
print("changed:", client.change(plan["id"], elevation_m=129))
client.remove(plan["id"])
client.remove(plan["id"])
print("plans added, in all:", plan_count() - before)

for line in client.log:
    print(" ", line)


created: Karasjok | plans added: 1
changed: {'id': 9, 'name': 'Karasjok', 'latitude': 69.47, 'longitude': 25.51, 'elevation_m': 129}
plans added, in all: 0
  POST /network/plans, attempt 1: ReadTimeout
  POST /network/plans, attempt 2: 201
  PATCH /network/plans/9, attempt 1: 200
  DELETE /network/plans/9, attempt 1: 204
  DELETE /network/plans/9, attempt 1: 404


### Where each part came from

| In the client | What it relies on | The section that showed it |
|---|---|---|
| `json=` on every body | a body written as JSON, with its `Content-Type` | What json= sends, and the same request from curl |
| one `Idempotency-Key` for all the attempts at a new plan | a repeated `POST` answered with its first result | An Idempotency-Key: one plan however often it is sent |
| `change` setting values with `PATCH` | a `PATCH` that sets values does no harm when it is repeated | Idempotent, and not |
| `status_code not in (204, 404)` in `remove` | a plan already removed is done | Removing a plan: DELETE |
| another attempt after a timeout, a lost connection or a `5xx` | the failures worth sending again | the **Errors and Retries** notebook |
| `raise_for_status()` once the attempts are over | a `422` or a `404` reported to the program, not sent again | A body the server refuses: 422 |

The new plan's first attempt lost its response, and the second, with the same key, got the `201`
back without creating another plan. The second `remove` met `404`, and the client took the plan for
removed, which it was. It never sends a `POST` without a key, the one request it could not safely send
again.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/13-sending-data-solutions.ipynb).

**1.** Send `/network/plans` a `POST` for a plan named Harstad, at latitude 68.80 and longitude 16.54,
and print the status code, the `Location` header and the body.


In [13]:
# your code here


**2.** Send a plan with no name, a latitude of 68.0 and a longitude of 200, and print each problem the
response lists.


In [14]:
# your code here


**3.** Use `PATCH` to give the plan from task 1 an `elevation_m` of 20, then `GET` the plan and print
it.


In [15]:
# your code here


**4.** Send a `PUT` with a whole plan to `/network/plans/999`, a plan that does not exist, and print the
status code and the error.


In [16]:
# your code here


**5.** Send `DELETE` to the plan from task 1 twice, and print both status codes and the status code of
a `GET` after them.


In [17]:
# your code here


**6.** Send the same `POST` for a plan named Karasjok, at latitude 69.47 and longitude 25.51, twice
with one `Idempotency-Key`. Print both status codes, the second response's `Idempotent-Replayed`
header, and how many plans are named Karasjok.


In [18]:
# your code here


## Common errors

### HTTPError: 415 Client Error: Unsupported Media Type for url: http://127.0.0.1:8765/network/plans


In [19]:
alta = {"name": "Alta", "latitude": 69.97, "longitude": 23.27}
response = requests.post(f"{BASE}/network/plans", alta, timeout=10)
response.raise_for_status()


HTTPError: 415 Client Error: Unsupported Media Type for url: http://127.0.0.1:8765/network/plans

The dictionary went in the second position, which belongs to `data`, not `json`, so requests sent it
as a form, and the practice API reads only JSON. A JSON string passed as `data=` is refused the same
way, because `data=` sets no `Content-Type` for a string. Name the argument, `json=`:


In [20]:
print(response.request.headers["Content-Type"], "|", response.request.body)
print(response.json()["error"])

response = requests.post(f"{BASE}/network/plans", json=alta, timeout=10)
print(response.status_code, response.reason)


application/x-www-form-urlencoded | name=Alta&latitude=69.97&longitude=23.27
send the body as JSON, with Content-Type: application/json
201 Created


### TypeError: Object of type date is not JSON serializable


In [21]:
plan = {"name": "Alta", "latitude": 69.97, "longitude": 23.27, "opens": date(2027, 6, 1)}
requests.post(f"{BASE}/network/plans", json=plan, timeout=10)


TypeError: Object of type date is not JSON serializable

JSON has strings, numbers, `true`, `false`, `null`, arrays and objects, and no dates, so `json=` could
not write a `date`, and raised before anything was sent. Send the date as the text the API documents,
here a date in ISO 8601 form, which `isoformat` writes:


In [22]:
plan["opens"] = date(2027, 6, 1).isoformat()
response = requests.post(f"{BASE}/network/plans", json=plan, timeout=10)
print(response.status_code, response.json())


201 {'id': 11, 'name': 'Alta', 'latitude': 69.97, 'longitude': 23.27, 'opens': '2027-06-01'}


### HTTPError: 422 Client Error: Unprocessable Content for url: http://127.0.0.1:8765/network/plans


In [23]:
row = {"name": "Lakselv", "latitude": "70.05", "longitude": "24.97"}       # a row read from a CSV file
response = requests.post(f"{BASE}/network/plans", json=row, timeout=10)
response.raise_for_status()


HTTPError: 422 Client Error: Unprocessable Content for url: http://127.0.0.1:8765/network/plans

Every value in a CSV row is text, as the **Headers and Content Types** notebook found, so the latitude
and the longitude went as strings, and the API wants numbers. The body says which fields were wrong.
Convert the values before sending them:


In [24]:
print([f"{problem['field']}: {problem['problem']}" for problem in response.json()["problems"]])

plan = {"name": row["name"], "latitude": float(row["latitude"]), "longitude": float(row["longitude"])}
print(requests.post(f"{BASE}/network/plans", json=plan, timeout=10).status_code)


['latitude: must be a number from -90 to 90', 'longitude: must be a number from -180 to 180']
201


### No error, and an elevation gone: a PUT that left a field out


In [25]:
created = requests.post(f"{BASE}/network/plans", json={"name": "Hasvik", "latitude": 70.49, "longitude": 22.14, "elevation_m": 20}, timeout=10)
hasvik_url = f"{BASE}{created.headers['Location']}"

response = requests.put(hasvik_url, json={"name": "Hasvik", "latitude": 70.48, "longitude": 22.14}, timeout=10)   # to correct the latitude
print(response.status_code, response.json())


200 {'id': 13, 'name': 'Hasvik', 'latitude': 70.48, 'longitude': 22.14}


The request was meant to correct the latitude. It succeeded, and the elevation is gone: a `PUT` body
is the whole plan, so a field it leaves out is not kept. Send only the change, with `PATCH`:


In [26]:
requests.put(hasvik_url, json={"name": "Hasvik", "latitude": 70.49, "longitude": 22.14, "elevation_m": 20}, timeout=10)
response = requests.patch(hasvik_url, json={"latitude": 70.48}, timeout=10)
print(response.status_code, response.json())


200 {'id': 13, 'name': 'Hasvik', 'latitude': 70.48, 'longitude': 22.14, 'elevation_m': 20}


### No error, and a change undone: two updates made from the same reading


In [27]:
office = requests.get(hasvik_url, timeout=10).json()      # two people read the same plan
field = requests.get(hasvik_url, timeout=10).json()

office["elevation_m"] = 25                                 # the office corrects the elevation
requests.put(hasvik_url, json=office, timeout=10)
field["latitude"] = 70.49                                  # and someone in the field corrects the latitude
requests.put(hasvik_url, json=field, timeout=10)

print(requests.get(hasvik_url, timeout=10).json())


{'id': 13, 'name': 'Hasvik', 'latitude': 70.49, 'longitude': 22.14, 'elevation_m': 20}


Both people read the plan, both changed their own field, and both sent the whole plan back. The
second `PUT` carried the elevation as it was when it was read, so it undid the office's change, and
neither request failed. Send the `ETag` from the reading back in an `If-Match` header, and the server
refuses a write to a plan that has changed since, with `412 Precondition Failed`:


In [28]:
first, second = requests.get(hasvik_url, timeout=10), requests.get(hasvik_url, timeout=10)

office = {**first.json(), "elevation_m": 25}
print("office:", requests.put(hasvik_url, json=office, headers={"If-Match": first.headers["ETag"]}, timeout=10).status_code)
field = {**second.json(), "latitude": 70.50}
refused = requests.put(hasvik_url, json=field, headers={"If-Match": second.headers["ETag"]}, timeout=10)
print("field: ", refused.status_code, refused.reason, "|", refused.json()["error"])


office: 200
field:  412 Precondition Failed | the plan has changed since that ETag was sent


The second write was refused instead of undoing the first. A client that meets `412` reads the plan
again, makes its change to what is there now, and sends it with the new `ETag`.


## Recap

- A request that changes something sends a body. `json=` writes it as JSON and sets
  `Content-Type: application/json`, and a server that cannot read a body's type answers `415`.
- `POST` creates, and `201 Created` names the new resource in `Location`. `PUT` replaces all of a
  resource, `PATCH` changes the fields it names, and `DELETE` removes, often with `204 No Content`.
- A `422` lists what is wrong with a body, so read its body before raising.
- `PUT` and `DELETE` are idempotent: sending one twice leaves the server as sending it once does, even
  when the second answer differs. `POST` is not, and `PATCH` is not promised to be.
- A `POST` whose response was lost cannot be told apart from one that failed. An `Idempotency-Key`,
  the same on every attempt, lets it be sent again without creating twice.
- `If-Match` with an `ETag` stops a change made from a stale copy from undoing another, with `412`.


## What is next

The **A Real Client** notebook. Every request here was written out in the cell that sent it. That
notebook gathers one API's requests, retries and errors into a small module you would reuse, with
tests that show it works.


---

&#8592; **Previous:** [Errors and Retries](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/12-errors-and-retries.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)  &nbsp;·&nbsp;  **Next:** [A Real Client](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/14-a-real-client.ipynb) &#8594;
